# MedQA EDA

Exploratory analysis for `GBaker/MedQA-USMLE-4-options`. This notebook checks dataset size, schema, label balance, option coverage, question length, and metadata fields before rows are normalized for CoCA training.

In [ ]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset("GBaker/MedQA-USMLE-4-options")
ds

In [ ]:
rows = []
for split, split_ds in ds.items():
    frame = split_ds.to_pandas()
    frame["split"] = split
    frame["question_chars"] = frame["question"].str.len()
    frame["question_words"] = frame["question"].str.split().str.len()
    frame["num_metamap_phrases"] = frame["metamap_phrases"].apply(lambda x: len(x) if isinstance(x, list) else 0)
    rows.append(frame)

medqa = pd.concat(rows, ignore_index=True)
medqa.head()

In [ ]:
summary = {
    "split_sizes": {split: len(split_ds) for split, split_ds in ds.items()},
    "answer_distribution": medqa.groupby(["split", "answer_idx"]).size().unstack(fill_value=0),
    "meta_info_distribution": medqa["meta_info"].value_counts(dropna=False).head(20),
    "question_length_by_split": medqa.groupby("split")[["question_chars", "question_words", "num_metamap_phrases"]].describe(),
}

summary